In [ ]:
import scanpy as sc
import decoupler as dc
import numpy as np
import anndata as ad
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import glob

In [ ]:
tma6 = sc.read("tma5_6_merged_tumour.h5ad")

In [ ]:
tma6_tumour = tma6[tma6.obs["Diagnosis"] == "Tumour"].copy()
# taking only the tumour-tissue cores

# Calculation of each cell-type's relative abundance

In [ ]:
## Note: Coarse refers to Tumour, Immune and Stromal compartments; whereas broad refers to the 17 lineage-specific types, and fine refers to the fine-grained annotated populations

In [ ]:
df = tma6_tumour.obs[[
    "PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"
]].copy()

for col in ["PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"]:
    df[col] = df[col].astype("category")

all_patients = df["PatientID"].cat.categories
all_broad   = df["decoupler"].cat.categories
all_fine    = df["decoupler_fine"].cat.categories
all_coarse  = df["decoupler_coarse"].cat.categories


In [ ]:
df = tma6_tumour.obs[["PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"]].copy()

# Ensure categories
for col in ["PatientID", "decoupler", "decoupler_fine", "decoupler_coarse"]:
    df[col] = df[col].astype("category")

all_patients = df["PatientID"].cat.categories
all_coarse   = df["decoupler_coarse"].cat.categories
all_broad    = df["decoupler"].cat.categories
all_fine     = df["decoupler_fine"].cat.categories


In [ ]:
## coarse on total cells

coarse_counts = (
    df.groupby(["PatientID", "decoupler_coarse"])
      .size().rename("n").reset_index()
)

patient_totals = (
    df.groupby("PatientID")
      .size().rename("denom").reset_index()
)

coarse_on_whole = (
    coarse_counts
    .merge(patient_totals, on="PatientID", how="left")
)
coarse_on_whole["fraction"] = coarse_on_whole["n"] / coarse_on_whole["denom"]

valid_coarse_whole = (
    pd.MultiIndex.from_product(
        [all_patients, all_coarse],
        names=["PatientID", "decoupler_coarse"]
    )
    .to_frame(index=False)
)

coarse_on_whole = (
    valid_coarse_whole
    .merge(coarse_on_whole, on=["PatientID", "decoupler_coarse"], how="left")
    .fillna({"n": 0, "fraction": 0})
)

coarse_on_whole.to_csv("tma6_tumour_decoupler_frac_coarse_on_whole.csv", index = False)

In [ ]:
broad_counts = (
    df.groupby(["PatientID", "decoupler"])
      .size().rename("n").reset_index()
)

broad_on_whole = (
    broad_counts
    .merge(patient_totals, on="PatientID", how="left")
)
broad_on_whole["fraction"] = broad_on_whole["n"] / broad_on_whole["denom"]

valid_broad_whole = (
    pd.MultiIndex.from_product(
        [all_patients, all_broad],
        names=["PatientID", "decoupler"]
    )
    .to_frame(index=False)
)

broad_on_whole = (
    valid_broad_whole
    .merge(broad_on_whole, on=["PatientID", "decoupler"], how="left")
    .fillna({"n": 0, "fraction": 0})
)

broad_on_whole.to_csv("tma6_tumour_decoupler_frac_broad_on_whole.csv", index = False)

In [ ]:
fine_mask = df["decoupler_fine"].astype(str) != df["decoupler"].astype(str)
fine_df = df[fine_mask]

fine_counts = (
    fine_df.groupby(["PatientID", "decoupler_fine"])
           .size().rename("n").reset_index()
)

fine_on_whole = (
    fine_counts
    .merge(patient_totals, on="PatientID", how="left")
)
fine_on_whole["fraction"] = fine_on_whole["n"] / fine_on_whole["denom"]


true_fine_labels = fine_df["decoupler_fine"].unique()

valid_fine_whole = (
    pd.MultiIndex.from_product(
        [all_patients, true_fine_labels],
        names=["PatientID", "decoupler_fine"]
    )
    .to_frame(index=False)
)

fine_on_whole = (
    valid_fine_whole
    .merge(fine_on_whole, on=["PatientID", "decoupler_fine"], how="left")
    .fillna({"n": 0, "fraction": 0})
)

fine_on_whole.to_csv("tma6_tumour_decoupler_frac_fine_on_whole.csv", index = False)

# Construction of the tumour cell specific pseudobulk

In [ ]:
adata = tma6_tumour.copy()

In [ ]:
adata.X = adata.layers['raw_counts'].copy()
adata.X[1:5, 1:5].toarray()
adata = adata[adata.obs["decoupler_coarse"] == "Tumour"].copy()

In [ ]:
adata

In [ ]:
obs_columns = ['decoupler_coarse', 'PatientID']

counts_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

meta = adata.obs[obs_columns]
combined_df = pd.concat([meta, counts_df], axis=1)

# Summed pseudobulk
pseudobulk_df = combined_df.groupby(obs_columns).sum()

cell_counts = combined_df.groupby(obs_columns).size()

pseudobulk_mean_df = pseudobulk_df.div(cell_counts, axis=0)
pb_long = (
    pseudobulk_mean_df
    .reset_index()
    .melt(
        id_vars=['decoupler_coarse', 'PatientID'],
        var_name='symbol',
        value_name='count'
    )
    .rename(columns={
        'decoupler_coarse': 'cell_type',
        'PatientID': 'sample'
    })
)


In [ ]:
pb_long.write_csv("tma6_tumour_decoupler_coarse_mean_bulk.csv")

## repeat similarly for TMA5 ###